<a href="https://colab.research.google.com/github/GHROTH-L/-ai-ml-training-/blob/main/Spaceship_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns #畫圖使用
%matplotlib inline
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import f_oneway
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.metrics import accuracy_score
#將dataframe 下載下來
from google.colab import files

In [2]:

from google.colab import files
# 上傳
uploaded = files.upload()  #for train
uploaded2 = files.upload()  #for test

Saving train.csv to train (3).csv


Saving test.csv to test (3).csv


In [70]:
train = pd.read_csv('train.csv')   # 最一開始
test = pd.read_csv('test.csv')

In [71]:
def fill_by_group_mode(df, group_col, target_col):
    group_mode = df.groupby(group_col)[target_col].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
    )
    df[target_col] = df[target_col].fillna(group_mode)
    return df

In [72]:
dfs = [train, test]
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for df in dfs:
    # 拆 PassengerId
    df[['Group', 'MemberNo']] = df['PassengerId'].str.split('_', expand=True)
    df['Group'] = pd.to_numeric(df['Group'], errors='coerce')
    df['MemberNo'] = pd.to_numeric(df['MemberNo'], errors='coerce')

    # 拆 Cabin
    df[['Deck', 'Deck_num', 'Deck_side']] = df['Cabin'].str.split('/', expand=True)
    df['Deck_num'] = pd.to_numeric(df['Deck_num'], errors='coerce')

    #冷凍艙
    df['CryoSleep'] = df['CryoSleep'].map({True:1, False:0})

    #補全VIP的數值
    df['VIP'] = df['VIP'].map({True:1, False:0})
    df['TotalSpending'] = df[spend_cols].fillna(0).sum(axis=1)
    df['VIP'] = df['VIP'].fillna(
    df['TotalSpending'] > 3000
                    )


##假定如果同個旅行團他的homeplanet 跟 destination Deck Deck_side 'CryoSleep' 應該要相同
for col in ['HomePlanet', 'Destination', 'Deck', 'Deck_side','CryoSleep']:
    train = fill_by_group_mode(train, 'Group', col)
    test = fill_by_group_mode(test, 'Group', col)




In [110]:
columns_missing = ['HomePlanet', 'CryoSleep', 'Destination',  'Age','VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'Deck', 'Deck_num', 'Deck_side','VRDeck']

# Manually impute missing values
for column in columns_missing:
    if column in ['HomePlanet', 'CryoSleep', 'Destination', 'VIP','Deck','Deck_side']:
        # For categorical columns, fill missing values with the most frequent value
        train[column].fillna(train[column].mode()[0], inplace=True)
        test[column].fillna(test[column].mode()[0], inplace=True)
    else:
        # For numerical columns, fill missing values with the mean
        train[column].fillna(train[column].mean(), inplace=True)
        test[column].fillna(test[column].mean(), inplace=True)

/tmp/ipykernel_17945/1875004084.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[column].fillna(train[column].mode()[0], inplace=True)
/tmp/ipykernel_17945/1875004084.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace

In [111]:
train.isnull().sum()

,0
PassengerId,0
HomePlanet,0
CryoSleep,0
Cabin,199
Destination,0
Age,0
VIP,0
RoomService,0
FoodCourt,0
ShoppingMall,0


In [116]:
features = [
    'HomePlanet',
    'Destination',
    'Deck',
    'Deck_side',
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa','VRDeck',
    'CryoSleep'
]
#進行資料預測
X = train[features]
y = train['Transported']

X = pd.get_dummies(X, drop_first=False)

In [117]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [118]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [119]:


y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))

Accuracy: 0.8033352501437608


In [121]:
model.fit(X, y)
X_test = test[features]
X_test = pd.get_dummies(X_test)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
pred = model.predict(X_test)

In [122]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": pred
})

In [124]:
submission.to_csv("submission.csv", index=False)
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>